# 5. Model Improvement – Dataset Diagnostics

In [2]:
import sys
import importlib

from pathlib import Path

project = Path("/home/aman-ajwa-fatima/Desktop/ForenSight-AI")

# Ensure project root is first in the import path
sys.path.insert(0, str(project))

# Remove incorrectly resolved src modules from the current kernel
for module_name in list(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        del sys.modules[module_name]

# Re-import from the correct project
import src
from src import config
from src.preprocessing.loaders import load_static, load_network

print("src location:", src.__file__)
print("Config location:", config.__file__)
print("Imports successful.")

src location: /home/aman-ajwa-fatima/Desktop/ForenSight-AI/src/__init__.py
Config location: /home/aman-ajwa-fatima/Desktop/ForenSight-AI/src/config.py
Imports successful.


In [3]:

import sklearn
import pandas as pd
import numpy as np

from src import config
from src.preprocessing.loaders import load_static, load_network

print("SCIKIT-LEARN VERSION:", sklearn.__version__)

datasets = [
    ("STATIC", load_static, config.STATIC_CSV),
    ("NETWORK", load_network, config.NETWORK_CSV)
]

for name, loader, path in datasets:

    bundle = loader(path)

    print("\n" + "=" * 60)
    print(name, "DATASET DIAGNOSTICS")
    print("=" * 60)

    print("Dataset shape:", bundle.X.shape)

    print("\nClass distribution:")
    print(
        bundle.y.value_counts().rename(
            index={0: "Benign", 1: "Malware"}
        )
    )

    print("\nClass percentages:")
    print(
        (bundle.y.value_counts(normalize=True) * 100).round(2)
    )

    print("\nMissing values:", int(bundle.X.isna().sum().sum()))

    print(
        "Infinite values:",
        int(np.isinf(
            bundle.X.select_dtypes(include=np.number).to_numpy()
        ).sum())
    )

    print("Number of features:", bundle.X.shape[1])

    if bundle.groups is not None:

        group_info = pd.DataFrame({
            "group": bundle.groups,
            "label": bundle.y
        })

        print("\nCapture files:", bundle.groups.nunique())

        print("\nCapture files by class:")
        print(
            pd.crosstab(
                group_info["group"],
                group_info["label"]
            ).gt(0).sum()
        )

        print(
            "\nMixed-label capture files:",
            int(
                (group_info.groupby("group")["label"].nunique() > 1)
                .sum()
            )
        )

print("\nDIAGNOSTICS COMPLETED")


SCIKIT-LEARN VERSION: 1.9.1

STATIC DATASET DIAGNOSTICS
Dataset shape: (662, 241)

Class distribution:
Label
Benign     531
Malware    131
Name: count, dtype: int64

Class percentages:
Label
0    80.21
1    19.79
Name: proportion, dtype: float64

Missing values: 0
Infinite values: 0
Number of features: 241

NETWORK DATASET DIAGNOSTICS
Dataset shape: (208004, 79)

Class distribution:
label_binary
Benign     167007
Malware     40997
Name: count, dtype: int64

Class percentages:
label_binary
0    80.29
1    19.71
Name: proportion, dtype: float64

Missing values: 0
Infinite values: 0
Number of features: 79

Capture files: 2113

Capture files by class:
label
0    1700
1     413
dtype: int64

Mixed-label capture files: 0

DIAGNOSTICS COMPLETED


In [4]:
bundle = load_network(config.NETWORK_CSV)

print("NETWORK DATASET SHAPE:", bundle.X.shape)

print("\nCLASS DISTRIBUTION:")
print(bundle.y.value_counts().rename(
    index={0: "Benign", 1: "Malware"}
))

print("\nCLASS PERCENTAGES:")
print((bundle.y.value_counts(normalize=True) * 100).round(2))

print("\nFEATURE COUNT:", bundle.X.shape[1])

print("\nMISSING VALUES:", bundle.X.isna().sum().sum())

print("\nINFINITE VALUES:", int(np.isinf(
    bundle.X.select_dtypes(include=np.number).to_numpy()
).sum()))

print("\nCAPTURE FILE COUNT:", bundle.groups.nunique())

NETWORK DATASET SHAPE: (208004, 79)

CLASS DISTRIBUTION:
label_binary
Benign     167007
Malware     40997
Name: count, dtype: int64

CLASS PERCENTAGES:
label_binary
0    80.29
1    19.71
Name: proportion, dtype: float64

FEATURE COUNT: 79

MISSING VALUES: 0

INFINITE VALUES: 0

CAPTURE FILE COUNT: 2113


In [5]:
# Network missing-value investigation

network_bundle = load_network(config.NETWORK_CSV)

missing_by_column = (
    network_bundle.X.isna()
    .sum()
    .sort_values(ascending=False)
)

print("COLUMNS WITH MISSING VALUES:")
print(missing_by_column[missing_by_column > 0])

print("\nROWS CONTAINING MISSING VALUES:")
print(int(network_bundle.X.isna().any(axis=1).sum()))

print("\nFEATURE DATA TYPES:")
print(network_bundle.X.dtypes.value_counts())

print("\nTOTAL MISSING CELLS:")
print(int(network_bundle.X.isna().sum().sum()))

COLUMNS WITH MISSING VALUES:
Series([], dtype: int64)

ROWS CONTAINING MISSING VALUES:
0

FEATURE DATA TYPES:
float64    79
Name: count, dtype: int64

TOTAL MISSING CELLS:
0


In [6]:

# Investigate the incomplete network record

network_bundle = load_network(config.NETWORK_CSV)

missing_mask = network_bundle.X.isna().any(axis=1)
missing_indices = network_bundle.X.index[missing_mask]

print("AFFECTED ROW INDEX:")
print(missing_indices.tolist())

print("\nAFFECTED ROW LABEL:")
print(network_bundle.y.loc[missing_indices])

print("\nAFFECTED ROW CAPTURE FILE:")
print(network_bundle.groups.loc[missing_indices])

print("\nMISSING FEATURE COUNT:")
print(network_bundle.X.loc[missing_indices].isna().sum(axis=1))

print("\nNON-MISSING VALUES IN AFFECTED ROW:")
print(network_bundle.X.loc[missing_indices].dropna(axis=1).T)


AFFECTED ROW INDEX:
[]

AFFECTED ROW LABEL:
Series([], Name: label_binary, dtype: int64)

AFFECTED ROW CAPTURE FILE:
Series([], Name: source_file, dtype: str)

MISSING FEATURE COUNT:
Series([], dtype: int64)

NON-MISSING VALUES IN AFFECTED ROW:
Empty DataFrame
Columns: []
Index: [Destination Port, Protocol, Flow Duration, Total Fwd Packets, Total Backward Packets, Total Length of Fwd Packets, Total Length of Bwd Packets, Fwd Packet Length Max, Fwd Packet Length Min, Fwd Packet Length Mean, Fwd Packet Length Std, Bwd Packet Length Max, Bwd Packet Length Min, Bwd Packet Length Mean, Bwd Packet Length Std, Flow Bytes/s, Flow Packets/s, Flow IAT Mean, Flow IAT Std, Flow IAT Max, Flow IAT Min, Fwd IAT Total, Fwd IAT Mean, Fwd IAT Std, Fwd IAT Max, Fwd IAT Min, Bwd IAT Total, Bwd IAT Mean, Bwd IAT Std, Bwd IAT Max, Bwd IAT Min, Fwd PSH Flags, Bwd PSH Flags, Fwd URG Flags, Bwd URG Flags, Fwd Header Length, Bwd Header Length, Fwd Packets/s, Bwd Packets/s, Min Packet Length, Max Packet Length, 

In [7]:

import pandas as pd

raw_network = pd.read_csv(
    config.NETWORK_CSV,
    low_memory=False
)

print("Original CSV shape:", raw_network.shape)

print("\nOriginal record at index 191071:")
display(raw_network.iloc[[191071]].T)

print("\nMissing values in original record:")
print(raw_network.iloc[191071].isna().sum())


Original CSV shape: (212600, 87)

Original record at index 191071:


,191071
Flow ID,172.217.7.10-10.42.0.211-443-54140-6
Source IP,10.42.0.211
Source Port,54140.0
Destination IP,172.217.7.10
Destination Port,443.0
...,...
Idle Max,0.0
Idle Min,0.0
Label,SMSMALWARE_BEANBOT
label_binary,1



Missing values in original record:
0


In [8]:
row = raw_network.iloc[[191071]]

print("Original row missing values:", row.isna().sum().sum())

print("\nColumns with missing values:")
print(row.columns[row.isna().iloc[0]].tolist())

print("\nOriginal row label:")
print(row.iloc[0].get("Label", row.iloc[0].get("label_binary")))

Original row missing values: 0

Columns with missing values:
[]

Original row label:
SMSMALWARE_BEANBOT


In [9]:

# Trace the processed record to its original CSV capture

network_bundle = load_network(config.NETWORK_CSV)

target_index = 191071

capture = network_bundle.groups.loc[target_index]

print("PROCESSED DATASET INDEX:", target_index)
print("CAPTURE FILE:", repr(capture))

print("\nORIGINAL CSV COLUMNS:")
print([
    col for col in raw_network.columns
    if "source" in col.lower() or "file" in col.lower()
])

print("\nLOADER LOG:")
print(network_bundle.log)

print("\nPROCESSED RECORD INDEX:")
print(network_bundle.X.index[network_bundle.X.index == target_index].tolist())


PROCESSED DATASET INDEX: 191071
CAPTURE FILE: 'SMSmalware-CSVs/SMSmalware/FakeNotify/07_13_2017-sms-fakenotify-F-secure-5e0da2476bb8ab81281bb61edfc84e18.pcap_ISCX.csv'

ORIGINAL CSV COLUMNS:
['Source IP', 'Source Port', 'source_file']

LOADER LOG:
{'source_file': '/home/aman-ajwa-fatima/Desktop/ForenSight-AI/datasets/network_analysis/cicandmal2017_flows.csv', 'label_column': 'label_binary (from capture folder / Label)', 'groups_available': True, 'n_capture_files': 2126, 'identifier_columns_removed': ['Flow ID', 'Source IP', 'Destination IP', 'Timestamp', 'Source Port', 'Label', 'source_file', 'label_binary'], 'raw_label_values': {'benign': 170000, 'malware': 42600}, 'rows_loaded': 212600, 'rows_dropped_missing_label': 0, 'dtypes_before': {'float64': 79}, 'missing_cells_after_coercion': 50, 'all_nan_columns_dropped': [], 'max_row_missing_fraction_before_filter': 0.6329113924050633, 'rows_dropped_excessive_missing_features': 1, 'duplicate_rows_removed': 4595, 'rows_final': 208004, 'n_fea

In [10]:

# Trace the affected record using its capture-file identifier

capture = network_bundle.groups.loc[191071]

identifier_columns = {
    "Flow ID",
    "Source IP",
    "Destination IP",
    "Timestamp",
    "Source Port",
    "Label",
    "source_file",
    "label_binary"
}

feature_columns = [
    col for col in raw_network.columns
    if col not in identifier_columns
]

# Find all original CSV records from this capture
capture_rows = raw_network[
    raw_network["source_file"].astype(str) == str(capture)
].copy()

# Apply the same numeric conversion used during preprocessing
numeric_capture = capture_rows[feature_columns].apply(
    pd.to_numeric,
    errors="coerce"
)

missing_counts = numeric_capture.isna().sum(axis=1)

print("CAPTURE FILE:", capture)
print("ORIGINAL ROWS IN THIS CAPTURE:", len(capture_rows))

print("\nROWS WITH MOST MISSING FEATURES:")
print(missing_counts.sort_values(ascending=False).head(10))

# Display the original values of the most incomplete record
if len(missing_counts):
    candidate_index = missing_counts.idxmax()

    print("\nMOST INCOMPLETE ORIGINAL CSV ROW:", candidate_index)
    print("MISSING FEATURE COUNT:", missing_counts.loc[candidate_index])

    print("\nORIGINAL RECORD:")
    display(capture_rows.loc[[candidate_index]].T)

    print("\nFEATURES THAT BECAME MISSING AFTER CONVERSION:")
    print(
        capture_rows.loc[candidate_index, feature_columns][
            numeric_capture.loc[candidate_index].isna()
        ]
    )


CAPTURE FILE: SMSmalware-CSVs/SMSmalware/FakeNotify/07_13_2017-sms-fakenotify-F-secure-5e0da2476bb8ab81281bb61edfc84e18.pcap_ISCX.csv
ORIGINAL ROWS IN THIS CAPTURE: 100

ROWS WITH MOST MISSING FEATURES:
194208    50
194200     0
194202     0
194201     0
194203     0
194204     0
194206     0
194205     0
194207     0
194209     0
dtype: int64

MOST INCOMPLETE ORIGINAL CSV ROW: 194208
MISSING FEATURE COUNT: 50

ORIGINAL RECORD:


,194208
Flow ID,182.254.5.234-10.42.0.211-80-59013-6
Source IP,10.42.0.211
Source Port,59013.0
Destination IP,182.254.5.234
Destination Port,80.0
...,...
Idle Max,NaN
Idle Min,NaN
Label,NaN
label_binary,1



FEATURES THAT BECAME MISSING AFTER CONVERSION:
Bwd IAT Max               NaN
Bwd IAT Min               NaN
Fwd PSH Flags             NaN
Bwd PSH Flags             NaN
Fwd URG Flags             NaN
Bwd URG Flags             NaN
Fwd Header Length         NaN
Bwd Header Length         NaN
Fwd Packets/s             NaN
Bwd Packets/s             NaN
Min Packet Length         NaN
Max Packet Length         NaN
Packet Length Mean        NaN
Packet Length Std         NaN
Packet Length Variance    NaN
FIN Flag Count            NaN
SYN Flag Count            NaN
RST Flag Count            NaN
PSH Flag Count            NaN
ACK Flag Count            NaN
URG Flag Count            NaN
CWE Flag Count            NaN
ECE Flag Count            NaN
Down/Up Ratio             NaN
Average Packet Size       NaN
Avg Fwd Segment Size      NaN
Avg Bwd Segment Size      NaN
Fwd Header Length.1       NaN
Fwd Avg Bytes/Bulk        NaN
Fwd Avg Packets/Bulk      NaN
Fwd Avg Bulk Rate         NaN
Bwd Avg Bytes/Bulk    

In [11]:
candidate_index = 194208

print("Original label:")
print(raw_network.loc[candidate_index, "label_binary"])

print("\nAvailable feature count:")
print(numeric_capture.loc[candidate_index].notna().sum())

print("\nMissing feature count:")
print(numeric_capture.loc[candidate_index].isna().sum())

print("\nCapture file:")
print(raw_network.loc[candidate_index, "source_file"])

Original label:
1

Available feature count:
29

Missing feature count:
50

Capture file:
SMSmalware-CSVs/SMSmalware/FakeNotify/07_13_2017-sms-fakenotify-F-secure-5e0da2476bb8ab81281bb61edfc84e18.pcap_ISCX.csv


In [12]:
network_bundle = load_network(config.NETWORK_CSV)

print("NETWORK CLEANING LOG")

for key, value in network_bundle.log.items():
    print(f"{key}: {value}")

print("\nFINAL MISSING VALUES:")
print(network_bundle.X.isna().sum().sum())

print("\nFINAL DATASET SHAPE:")
print(network_bundle.X.shape)

print("\nCLASS DISTRIBUTION:")
print(network_bundle.y.value_counts())

NETWORK CLEANING LOG
source_file: /home/aman-ajwa-fatima/Desktop/ForenSight-AI/datasets/network_analysis/cicandmal2017_flows.csv
label_column: label_binary (from capture folder / Label)
groups_available: True
n_capture_files: 2126
identifier_columns_removed: ['Flow ID', 'Source IP', 'Destination IP', 'Timestamp', 'Source Port', 'Label', 'source_file', 'label_binary']
raw_label_values: {'benign': 170000, 'malware': 42600}
rows_loaded: 212600
rows_dropped_missing_label: 0
dtypes_before: {'float64': 79}
missing_cells_after_coercion: 50
all_nan_columns_dropped: []
max_row_missing_fraction_before_filter: 0.6329113924050633
rows_dropped_excessive_missing_features: 1
duplicate_rows_removed: 4595
rows_final: 208004
n_features: 79
missing_cells_final: 0
class_counts: {'benign': 167007, 'malware': 40997}
n_capture_files_final: 2113

FINAL MISSING VALUES:
0

FINAL DATASET SHAPE:
(208004, 79)

CLASS DISTRIBUTION:
label_binary
0    167007
1     40997
Name: count, dtype: int64


In [13]:
from src.models.train import make_models

for improved in [False, True]:

    print("\nIMPROVED MODE:", improved)

    models = make_models(improved)

    for name, (pipeline, grid, search_type) in models.items():

        print(f"\n{name}")
        print("Search:", search_type)

        model = pipeline.named_steps["model"]

        print("Supports ccp_alpha:",
              "ccp_alpha" in model.get_params())

        print("Parameters:", list(grid.keys()))


IMPROVED MODE: False

Logistic Regression
Search: grid
Supports ccp_alpha: False
Parameters: ['model__C']

Random Forest
Search: random
Supports ccp_alpha: True
Parameters: ['model__n_estimators', 'model__max_depth', 'model__min_samples_split', 'model__min_samples_leaf', 'model__max_features', 'model__ccp_alpha']

Gradient Boosting
Search: random
Supports ccp_alpha: True
Parameters: ['model__n_estimators', 'model__learning_rate', 'model__max_depth', 'model__min_samples_split', 'model__min_samples_leaf', 'model__ccp_alpha']

IMPROVED MODE: True

Logistic Regression
Search: grid
Supports ccp_alpha: False
Parameters: ['model__C']

Random Forest
Search: random
Supports ccp_alpha: True
Parameters: ['model__n_estimators', 'model__max_depth', 'model__min_samples_split', 'model__min_samples_leaf', 'model__max_features', 'model__ccp_alpha']

Gradient Boosting
Search: random
Supports ccp_alpha: True
Parameters: ['model__n_estimators', 'model__learning_rate', 'model__max_depth', 'model__min_samp